# Activation Functions: From Sigmoid to GELU

Wiki reference for [activation functions](https://ml-viz-ruby.vercel.app/wiki/activation-functions).

**The idea in one sentence.** The activation function decides whether gradients survive depth:
**saturating** functions (sigmoid, tanh) shrink the gradient by $\le 0.25$ per layer and
vanish in deep nets, while **ReLU-family** functions keep the gradient at 1 on the active side
— at the cost of a possible **dead-unit** problem that smooth variants (LeakyReLU, GELU, SiLU)
fix.

We implement the activation zoo and their derivatives from scratch, **validate the dead-ReLU
problem, the vanishing-gradient scaling, and numerically stable softmax**, then cover the
gotchas.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
plt.style.use('dark_background')
rng = np.random.default_rng(0)

## 1 — The activation zoo and their derivatives

In [ ]:
sigmoid = lambda z: 1/(1+np.exp(-z))
tanh    = lambda z: np.tanh(z)
relu    = lambda z: np.maximum(0, z)
lrelu   = lambda z, a=0.1: np.where(z>0, z, a*z)
elu     = lambda z, a=1.0: np.where(z>0, z, a*(np.exp(z)-1))
gelu    = lambda z: z*norm.cdf(z)
silu    = lambda z: z*sigmoid(z)   # Swish

# derivatives
d_sigmoid = lambda z: sigmoid(z)*(1-sigmoid(z))
d_tanh    = lambda z: 1-np.tanh(z)**2
d_relu    = lambda z: (z>0).astype(float)

z = np.linspace(-5, 5, 400)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for fn, name in [(sigmoid,'sigmoid'),(tanh,'tanh'),(relu,'relu'),(lrelu,'leaky'),(gelu,'gelu'),(silu,'silu')]:
    ax[0].plot(z, fn(z), label=name)
ax[0].set_title('Activations'); ax[0].legend(fontsize=8); ax[0].axhline(0,color='gray',lw=0.5); ax[0].axvline(0,color='gray',lw=0.5)
for fn, name in [(d_sigmoid,"sigmoid'"),(d_tanh,"tanh'"),(d_relu,"relu'")]:
    ax[1].plot(z, fn(z), label=name)
ax[1].axhline(0.25, color='#f87171', ls='--', label='sigmoid max=0.25')
ax[1].set_title('Derivatives (note sigmoid ceiling)'); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 2 — The dead ReLU problem

A neuron pushed negative for every input outputs 0 with gradient 0 — and can never recover.

In [ ]:
X = rng.normal(0, 1, (1000, 4))
# A unit with a large negative bias: pre-activation is negative for ~all inputs
w = rng.normal(0, 0.5, 4)
b_dead = -6.0
z_dead = X @ w + b_dead
print(f"ReLU unit with bias {b_dead}: active on {(z_dead>0).mean()*100:.1f}% of inputs")
print(f"  -> gradient is 0 on {(z_dead<=0).mean()*100:.1f}% of inputs (this unit is effectively dead)")

# Leaky ReLU keeps a small gradient alive
grad_relu  = d_relu(z_dead).mean()
grad_leaky = np.where(z_dead>0, 1.0, 0.1).mean()
print(f"\nMean gradient   ReLU: {grad_relu:.3f}   LeakyReLU: {grad_leaky:.3f}")
print("LeakyReLU still passes gradient, so the unit can recover.")

### Validate: the dead-ReLU problem

A ReLU unit with a large negative bias is off for almost every input, so its gradient is ~0
and it can never recover — it is *dead*. LeakyReLU keeps a small gradient on the negative side,
so the same unit stays alive. We confirm.

In [ ]:
print(f'dead-ReLU mean gradient : {grad_relu:.3f}')
print(f'LeakyReLU mean gradient : {grad_leaky:.3f}')
assert grad_relu < 0.05, 'the dead ReLU unit passes almost no gradient'
assert grad_leaky > grad_relu, 'LeakyReLU keeps gradient alive so the unit can recover'
print('\n✅ ReLU units can die; leaky/smooth variants keep a gradient path open')

## 3 — Saturation -> vanishing gradients through depth

Backprop multiplies activation derivatives layer by layer. Sigmoid's 0.25 ceiling shrinks the signal geometrically; ReLU's 1.0 preserves it.

In [ ]:
depths = np.arange(1, 21)
# Best case: every derivative at its maximum
sig_grad  = 0.25 ** depths
relu_grad = 1.0  ** depths

plt.figure(figsize=(8, 4))
plt.semilogy(depths, sig_grad, 'o-', label='sigmoid (0.25^depth)', color='#f87171')
plt.semilogy(depths, relu_grad, 's-', label='relu (1.0^depth)', color='#34d399')
plt.xlabel('depth (layers)'); plt.ylabel('gradient scale (log)')
plt.title('Why saturating activations kill deep nets'); plt.legend()
plt.tight_layout(); plt.show()
print(f"At depth 10: sigmoid scales gradient by {0.25**10:.2e}, relu by {1.0**10:.0f}")

### Validate: saturating activations vanish with depth

Chain-rule multiplies one derivative per layer. Sigmoid's derivative peaks at $0.25$, so the
best-case gradient scales as $0.25^{\text{depth}}$ — astronomically small in deep nets — while
ReLU's is $1^{\text{depth}} = 1$. We confirm the collapse.

In [ ]:
print(f'depth 10: sigmoid scale {sig_grad[9]:.2e}, relu scale {relu_grad[9]:.0f}')
assert sig_grad[-1] < 1e-8, 'sigmoid gradient vanishes exponentially with depth'
assert relu_grad[-1] == 1.0, 'ReLU preserves gradient magnitude with depth'
print('\n✅ saturating activations kill deep nets; ReLU-family fixed the vanishing gradient')

## 4 — Numerically stable softmax

Softmax is shift-invariant, so subtract the max logit before exponentiating to avoid overflow.

In [ ]:
def softmax_naive(z):
    e = np.exp(z); return e / e.sum()

def softmax_stable(z):
    e = np.exp(z - z.max()); return e / e.sum()

logits = np.array([1000.0, 1001.0, 1002.0])  # huge logits
print("naive: ", softmax_naive(logits), "<- overflow to nan")
print("stable:", softmax_stable(logits).round(4), "<- correct")
assert np.isclose(softmax_stable(logits).sum(), 1.0)

### Validate: numerically stable softmax

Softmax is shift-invariant, so subtracting the max before exponentiating changes nothing
mathematically but prevents overflow. On huge logits the naive version overflows to NaN while
the stable version is correct. We confirm.

In [ ]:
print(f'naive : {softmax_naive(logits)}')
print(f'stable: {softmax_stable(logits).round(4)}')
assert np.isnan(softmax_naive(logits)).any(), 'naive softmax overflows to NaN on huge logits'
assert np.isclose(softmax_stable(logits).sum(), 1.0), 'stable softmax is a valid distribution'
print('\n✅ subtract the max before exp — same math, no overflow')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **sigmoid/tanh in deep nets** | gradient vanishes as $0.25^{\text{depth}}$ (verified) |
| **dead ReLU** | a unit off for all inputs never recovers (verified) — use leaky/GELU |
| **naive softmax** | overflows to NaN on large logits (verified) — subtract the max |
| **ReLU hard dead zone** | zero gradient for z<0 (demo) — smooth variants avoid it |
| **choosing an activation** | GELU/SiLU are strong defaults; match to architecture |

Demo: ReLU has a hard dead zone; GELU stays smooth on the negative side.

In [ ]:
# The dead-zone gotcha precisely: ReLU's gradient is EXACTLY zero for every negative input, so
# a unit pushed negative is stuck. Smooth activations (GELU, SiLU) stay non-zero on the
# negative side, which is why modern transformers use them. We confirm the difference.
z_neg = np.linspace(-3, -0.1, 50)
assert (d_relu(z_neg) == 0).all(), 'ReLU has a hard dead zone: gradient is exactly 0 for z<0'
assert (np.abs(gelu(z_neg)) > 0).any(), 'GELU stays smooth and non-zero on the negative side'
print(f'ReLU gradient on negatives : all zero  -> dead zone')
print(f'GELU output on negatives   : nonzero (e.g. gelu(-1)={gelu(np.array([-1.0]))[0]:.3f}) -> gradient flows')
print('\nSmooth activations (GELU/SiLU) avoid the hard dead zone -> the default in modern transformers.')

## ✏️ Your turn

**Task A — tanh approximation of GELU:** Implement the popular tanh approximation $0.5z(1 + \tanh[\sqrt{2/\pi}(z + 0.044715 z^3)])$ and show its max absolute error vs the exact $z\,\Phi(z)$ over $z \in [-5,5]$.

**Task B — empirical dead-unit rate:** For a layer of 256 ReLU units with He-initialized weights on standard-normal input, measure what fraction of units are dead (zero output on the whole batch). Repeat with a too-large negative bias and show the rate spikes.

In [ ]:
def gelu_tanh(z):
    # TODO(you): implement the tanh approximation of GELU
    return ...

z = np.linspace(-5, 5, 1000)
approx = gelu_tanh(z)
if approx is not None:
    err = np.abs(approx - gelu(z)).max()
    print(f"Max abs error of tanh-GELU vs exact: {err:.6f}")

<details><summary>Solution — Task A</summary>

```python
def gelu_tanh(z):
    return 0.5*z*(1 + np.tanh(np.sqrt(2/np.pi)*(z + 0.044715*z**3)))
# Max error is ~0.0003 — why the approximation is used interchangeably in practice.
```
</details>

## Key takeaways

- **Saturating activations vanish with depth:** sigmoid scales the gradient by $\le 0.25$ per
  layer (verified) — the reason deep nets moved to ReLU.
- **ReLU preserves gradient** on the active side but units can **die** (verified) — leaky/GELU/
  SiLU keep a path open (demo).
- **Stable softmax:** subtract the max before exponentiating — same math, no overflow
  (verified).
- **Smooth activations** (GELU, SiLU) are the modern default: no hard dead zone.